In [12]:
import pandas as pd
import numpy as np
import folium
from folium.plugins import HeatMap

df = pd.read_csv('Divar.csv') 
df.shape

C:\Users\Mahyar\AppData\Local\Temp\ipykernel_17692\1197405370.py:6: DtypeWarning: Columns (0: rent_to_single, 1: floor, 2: total_floors_count, 3: extra_person_capacity) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('Divar.csv')


(1000000, 61)

In [13]:
print(df[['location_latitude', 'location_longitude']].isna().sum())
print(df.shape)

location_latitude     344392
location_longitude    344392
dtype: int64
(1000000, 61)


<h2 align=right style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
بررسی صحت داده‌های مکانی
</font>
</h2>

<p dir=rtl style="direction: rtl; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazir" size=3>
پیش از رسم نقشه‌ی حرارتی، بخشی از رکوردهای دیتاست (حدود ۳۴ درصد) فاقد مختصات جغرافیایی معتبر بودند. از آنجا که این نقشه صرفاً بر اساس رکوردهای دارای مختصات رسم می‌شود، لازم بود بررسی شود که آیا حذف این رکوردها می‌تواند نتیجه‌ی نهایی را دچار اریبی (انحراف) کند یا خیر.
    <br><br>
    برای این منظور، ده شهر پرتکرار را در دو گروه مقایسه کردیم: گروه رکوردهایی که مختصات جغرافیایی نداشتند و گروه رکوردهایی که مختصات معتبر داشتند. نتایج نشان داد که توزیع شهرها در این دو گروه یکسان نیست. برای نمونه، تهران در گروه دارای مختصات سهم بیشتری (حدود ۲۲ درصد) نسبت به گروه بدون مختصات (حدود ۱۴ درصد) داشت، در حالی که مشهد و شیراز روند معکوسی داشتند و سهم بیشتری در گروه بدون مختصات نشان دادند.
    <br><br>
    این یافته نشان می‌دهد که برخی شهرها مانند مشهد و شیراز به‌طور نسبی مختصات کمتری ثبت کرده‌اند و ممکن است تراکم واقعی آن‌ها در نقشه‌ی حرارتی نهایی کمی کمتر از واقعیت نمایش داده شود. با این حال، از آنجا که تهران در هر دو گروه با فاصله‌ی زیادی نسبت به سایر شهرها در رتبه‌ی نخست قرار دارد، این محدودیت تأثیری بر نتیجه‌ی کلی (شناسایی تهران به‌عنوان پرتراکم‌ترین منطقه) ندارد و صرفاً باید در تفسیر جایگاه شهرهای رتبه‌های بعدی با احتیاط بیشتری عمل کرد.
</font>
</p>

In [14]:
nan_mask = df['location_latitude'].isna() | df['location_longitude'].isna()

print("۱۰ شهر پرتکرار در بین رکوردهای بدون مختصات:")
print(df.loc[nan_mask, 'city_slug'].value_counts(normalize=True).head(10))
print()
print("۱۰ شهر پرتکرار در بین رکوردهای دارای مختصات:")
print(df.loc[~nan_mask, 'city_slug'].value_counts(normalize=True).head(10))

۱۰ شهر پرتکرار در بین رکوردهای بدون مختصات:
city_slug
tehran          0.138860
mashhad         0.091971
shiraz          0.049926
karaj           0.038706
isfahan         0.035370
tabriz          0.031296
ahvaz           0.029731
qom             0.022376
bandar-abbas    0.018052
kermanshah      0.018000
Name: proportion, dtype: float64

۱۰ شهر پرتکرار در بین رکوردهای دارای مختصات:
city_slug
tehran               0.218243
mashhad              0.056982
karaj                0.054967
isfahan              0.037785
shiraz               0.030425
tabriz               0.025743
andisheh-new-town    0.023497
rasht                0.018841
kermanshah           0.016548
qom                  0.014342
Name: proportion, dtype: float64


In [15]:
map_data = df.dropna(subset=['location_latitude', 'location_longitude']).copy()
print("تعداد ردیف‌های معتبر برای نقشه:", map_data.shape)

heat_data = map_data[['location_latitude', 'location_longitude']].values.tolist()
print("تعداد نقاط استفاده‌شده برای رسم:", len(heat_data))

تعداد ردیف‌های معتبر برای نقشه: (655608, 61)
تعداد نقاط استفاده‌شده برای رسم: 655608


In [16]:
center_lat = map_data['location_latitude'].mean()
center_lon = map_data['location_longitude'].mean()

m = folium.Map(location=[center_lat, center_lon], zoom_start=6, tiles='CartoDB positron')
HeatMap(heat_data, radius=8, blur=10, max_zoom=13).add_to(m)
m.save('heatmap.html')
print("نقشه ذخیره شد در: heatmap.html")

نقشه ذخیره شد در: heatmap.html


<h2 align=right style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
جمع‌بندی و نتیجه‌گیری
</font>
</h2>

<p dir=rtl style="direction: rtl; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazir" size=3>
با رسم نقشه‌ی حرارتی بر اساس مختصات جغرافیایی آگهی‌ها، مشخص شد که تهران به‌طور بی‌رقیب پرتراکم‌ترین منطقه از نظر تعداد آگهی‌های ثبت‌شده در دیوار است؛ به‌گونه‌ای که در نقشه به‌صورت یک نقطه‌ی کاملاً متمایز و پررنگ در مقایسه با سایر مناطق کشور دیده می‌شود.
    <br><br>
    پس از تهران، مناطق فلات مرکزی ایران شامل اصفهان و شیراز و همچنین کلان‌شهرهایی مانند مشهد، تراکم متوسط تا نسبتاً بالایی را نشان می‌دهند، هرچند با فاصله‌ی زیادی نسبت به تهران. مناطق حاشیه‌ای و مرزی کشور، به‌ویژه در شرق و جنوب شرق، کمترین تراکم آگهی را دارند.
    <br><br>
    لازم به ذکر است که به دلیل فقدان مختصات جغرافیایی در بخشی از داده‌ها، ممکن است تراکم واقعی برخی شهرها مانند مشهد و شیراز اندکی بیشتر از آنچه در نقشه مشاهده می‌شود باشد؛ با این حال، این محدودیت جایگاه تهران به‌عنوان پرتراکم‌ترین منطقه را تحت‌تأثیر قرار نمی‌دهد.
</font>
</p>